In [61]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import catboost
from catboost import CatBoostClassifier
import optuna 
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, classification_report
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb

In [28]:
train = pd.read_parquet('train.parquet')
test = pd.read_parquet('test.parquet')
val = pd.read_parquet('val.parquet')
target1 = 'Target1'
target2 = 'Target2'

In [29]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 591285 entries, 963695 to 636609
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Age             551055 non-null  float64
 1   Ind_Household   591285 non-null  object 
 2   Age_group       591285 non-null  object 
 3   District        591285 non-null  object 
 4   Region          591285 non-null  object 
 5   Lifetime        583726 non-null  float64
 6   Income          591285 non-null  int64  
 7   Segment         591285 non-null  object 
 8   Ind_deposit     591285 non-null  object 
 9   Ind_email       591285 non-null  object 
 10  Ind_phone       591285 non-null  object 
 11  Ind_salary      591285 non-null  object 
 12  trans_6_month   591285 non-null  float64
 13  trans_9_month   591285 non-null  float64
 14  trans_12_month  591285 non-null  float64
 15  amont_trans     591285 non-null  int64  
 16  amont_day_from  591285 non-null  int64  
 17  trans_3_mo

In [30]:
train = train.drop(columns=['Ind_Household', 'Region', 'Ind_deposit', 'Ind_email', 'Ind_phone', 'Ind_salary', 'trans_6_month', 'trans_9_month', 'trans_12_month', 'amont_trans'])
val = val.drop(columns=['Ind_Household', 'Region', 'Ind_deposit', 'Ind_email', 'Ind_phone', 'Ind_salary', 'trans_6_month', 'trans_9_month', 'trans_12_month', 'amont_trans'])
test = test.drop(columns=['Ind_Household', 'Region', 'Ind_deposit', 'Ind_email', 'Ind_phone', 'Ind_salary', 'trans_6_month', 'trans_9_month', 'trans_12_month', 'amont_trans'])




In [31]:
train

,Age,Age_group,District,Lifetime,Income,Segment,amont_day_from,trans_3_month,Gender,Target1,Target2
963695,73.0,senior,09,4.0,53,Gold,24,961.780000,M,1,1
759675,37.0,middle,52,4.0,58,Silver,28,935.174529,F,1,0
245217,NaN,unknown,03,16.0,49,Silver,15,991.400000,U,0,0
71316,51.0,middle,53,5.0,55,Tin,15,969.010000,F,0,0
256377,44.0,middle,03,7.0,40,Gold,16,927.500000,F,0,0
...,...,...,...,...,...,...,...,...,...,...,...
906602,52.0,middle,49,8.0,55,Gold,14,960.200000,M,0,0
829155,62.0,senior,32,2.0,46,Gold,13,984.400000,F,0,0
459940,69.0,senior,42,8.0,44,Platinum,14,974.060000,M,0,0
296417,76.0,senior,50,8.0,50,Gold,14,929.980000,M,0,0


# Catboost

In [32]:
cat_features = train.select_dtypes(include=['object', 'category']).columns.tolist()

In [33]:
cat_features

['Age_group', 'District', 'Segment', 'Gender']

In [34]:
model = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=3, random_seed=42)
model.fit(train.drop(columns=[target1, target2]), train[target1], cat_features=cat_features, eval_set = (val.drop(columns=[target1, target2]), val[target1]), verbose=100)

0:	learn: 0.5969396	test: 0.5967728	best: 0.5967728 (0)	total: 113ms	remaining: 1m 52s
100:	learn: 0.2549963	test: 0.2548214	best: 0.2548214 (100)	total: 7.37s	remaining: 1m 5s
200:	learn: 0.2458418	test: 0.2450956	best: 0.2450956 (200)	total: 14.6s	remaining: 58.1s
300:	learn: 0.2408380	test: 0.2397619	best: 0.2397619 (300)	total: 21.6s	remaining: 50.3s
400:	learn: 0.2375018	test: 0.2363773	best: 0.2363773 (400)	total: 28.8s	remaining: 43.1s
500:	learn: 0.2348566	test: 0.2335464	best: 0.2335464 (500)	total: 36.2s	remaining: 36.1s
600:	learn: 0.2327023	test: 0.2313732	best: 0.2313732 (600)	total: 43.6s	remaining: 29s
700:	learn: 0.2311042	test: 0.2297689	best: 0.2297689 (700)	total: 50.7s	remaining: 21.6s
800:	learn: 0.2296540	test: 0.2282846	best: 0.2282846 (800)	total: 58.1s	remaining: 14.4s
900:	learn: 0.2283245	test: 0.2268582	best: 0.2268582 (900)	total: 1m 5s	remaining: 7.17s
999:	learn: 0.2272581	test: 0.2257500	best: 0.2257500 (999)	total: 1m 12s	remaining: 0us

bestTest = 0.22

In [35]:
print(f"TRAIN : {roc_auc_score(train[target1], model.predict_proba(train.drop(columns=[target1, target2]))[:, 1])}")
print(f"VAL : {roc_auc_score(val[target1], model.predict_proba(val.drop(columns=[target1, target2]))[:, 1])}")
print(f"TEST : {roc_auc_score(test[target1], model.predict_proba(test.drop(columns=[target1, target2]))[:, 1])}")

TRAIN : 0.9484027500049939
VAL : 0.9478111656994044
TEST : 0.9483679453270111


In [36]:
print(classification_report(test[target1], model.predict(test.drop(columns=[target1, target2]))))

              precision    recall  f1-score   support

           0       0.92      0.97      0.94    148281
           1       0.88      0.73      0.80     48815

    accuracy                           0.91    197096
   macro avg       0.90      0.85      0.87    197096
weighted avg       0.91      0.91      0.91    197096



# Decision tree

In [37]:
for col in train.select_dtypes(include='number').columns:
    train[col].fillna(train[col].median(), inplace=True)

for col in train.select_dtypes(include='object').columns:
    train[col].fillna(train[col].mode()[0], inplace=True)




for col in val.select_dtypes(include='number').columns:
    val[col].fillna(val[col].median(), inplace=True)

for col in val.select_dtypes(include='object').columns:
    val[col].fillna(val[col].mode()[0], inplace=True)



for col in test.select_dtypes(include='number').columns:
    test[col].fillna(test[col].median(), inplace=True)

for col in test.select_dtypes(include='object').columns:
    test[col].fillna(test[col].mode()[0], inplace=True)

    


train_new = pd.get_dummies(train, drop_first=True)
val_new = pd.get_dummies(val, drop_first=True)
test_new = pd.get_dummies(test, drop_first=True)

/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_22098/2452799383.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(train[col].median(), inplace=True)
/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_22098/2452799383.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values 

In [40]:
model = DecisionTreeClassifier(random_state=42)
model.fit(train_new.drop(columns=['Target1', 'Target2']), train_new['Target1'])

DecisionTreeClassifier(random_state=42)

In [41]:
print(f"TRAIN : {roc_auc_score(train_new[target1], model.predict_proba(train_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"VAL : {roc_auc_score(val_new[target1], model.predict_proba(val_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"TEST : {roc_auc_score(test_new[target1], model.predict_proba(test_new.drop(columns=[target1, target2]))[:, 1])}")

TRAIN : 1.0
VAL : 0.8939325338257519
TEST : 0.8940170376275304


In [44]:
y_pred = model.predict(test_new.drop(columns=['Target1', 'Target2']))
y_proba = model.predict_proba(test_new.drop(columns=['Target1', 'Target2']))[:, 1]

accuracy = accuracy_score(test_new['Target1'], y_pred)

print(f'Accuracy: {accuracy:.3f}')

Accuracy: 0.921


In [45]:
f1 = f1_score(test_new['Target1'], y_pred)
print(f'F1 Score: {f1:.3f}')

F1 Score: 0.841


# Logistict Regression 

In [46]:
model = LogisticRegression()
model.fit(train_new.drop(columns=[target1, target2]), train_new[target1])

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [47]:
print(f"TRAIN : {roc_auc_score(train_new[target1], model.predict_proba(train_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"VAL : {roc_auc_score(val_new[target1], model.predict_proba(val_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"TEST : {roc_auc_score(test_new[target1], model.predict_proba(test_new.drop(columns=[target1, target2]))[:, 1])}")

TRAIN : 0.8172962381560673
VAL : 0.8171706981658842
TEST : 0.8183335778266467


In [48]:
print(f"TRAIN : {f1_score(train_new[target1], model.predict(train_new.drop(columns=[target1, target2])))}")
print(f"VAL : {f1_score(val_new[target1], model.predict(val_new.drop(columns=[target1, target2])))}")
print(f"TEST : {f1_score(test_new[target1], model.predict(test_new.drop(columns=[target1, target2])))}")

TRAIN : 0.5189905642558573
VAL : 0.5186799039389963
TEST : 0.5206837186424004


In [50]:
y_pred = model.predict(test_new.drop(columns=[target1, target2]))

accuracy = accuracy_score(test_new[target1], y_pred)
precision = precision_score(test_new[target1], y_pred)
recall = recall_score(test_new[target1], y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Accuracy: 0.80
Precision: 0.65
Recall: 0.43


# Random Forest

In [53]:
model = RandomForestClassifier(random_state=42)
model.fit(train_new.drop(columns=['Target1', 'Target2']), train_new['Target1'])

RandomForestClassifier(random_state=42)

In [54]:
print(f"TRAIN : {roc_auc_score(train_new[target1], model.predict_proba(train_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"VAL : {roc_auc_score(val_new[target1], model.predict_proba(val_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"TEST : {roc_auc_score(test_new[target1], model.predict_proba(test_new.drop(columns=[target1, target2]))[:, 1])}")

TRAIN : 1.0
VAL : 0.9786284438340552
TEST : 0.9792054709268052


In [55]:
y_pred = model.predict(test_new.drop(columns=['Target1', 'Target2']))
y_proba = model.predict_proba(test_new.drop(columns=['Target1', 'Target2']))[:, 1]

accuracy = accuracy_score(test_new['Target1'], y_pred)

print(f'Accuracy: {accuracy:.3f}')

Accuracy: 0.935


In [56]:
f1 = f1_score(test_new['Target1'], y_pred)
print(f'F1 Score: {f1:.3f}')

F1 Score: 0.860


# XGBoost

In [59]:
dtrain = xgb.DMatrix(train_new.drop(columns=[target1, target2]), label=train_new[target1])
dval = xgb.DMatrix(val_new.drop(columns=[target1, target2]), label=val_new[target1])
dtest = xgb.DMatrix(test_new.drop(columns=[target1, target2]), label=test_new[target1])

params = {
    'eta': 0.1,
    'max_depth': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

evals = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(params, 
                  dtrain, 
                  num_boost_round=100, 
                  evals=evals,
                  early_stopping_rounds=10,
                  verbose_eval=10)

[0]	train-rmse:0.41623	val-rmse:0.41622
[10]	train-rmse:0.31973	val-rmse:0.31964
[20]	train-rmse:0.29546	val-rmse:0.29543
[30]	train-rmse:0.28665	val-rmse:0.28665
[40]	train-rmse:0.28386	val-rmse:0.28386
[50]	train-rmse:0.28241	val-rmse:0.28240
[60]	train-rmse:0.28168	val-rmse:0.28169
[70]	train-rmse:0.28119	val-rmse:0.28120
[80]	train-rmse:0.28081	val-rmse:0.28082
[90]	train-rmse:0.28052	val-rmse:0.28054
[99]	train-rmse:0.28026	val-rmse:0.28029


In [60]:
y_pred_prob = model.predict(dtest)
y_pred = (y_pred_prob > 0.5).astype(int)

def print_metrics(y_true, y_pred, y_pred_prob):
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_pred_prob))

print("\nTest set metrics:")
print_metrics(test_new[target1], y_pred, y_pred_prob)

print("\nTrain set metrics:")
y_train_pred_prob = model.predict(dtrain)
y_train_pred = (y_train_pred_prob > 0.5).astype(int)
print_metrics(train_new[target1], y_train_pred, y_train_pred_prob)

print("\nValidation set metrics:")
y_val_pred_prob = model.predict(dval)
y_val_pred = (y_val_pred_prob > 0.5).astype(int)
print_metrics(val_new[target1], y_val_pred, y_val_pred_prob)


Test set metrics:
Accuracy: 0.8966747168892316
Precision: 0.8702691446717684
Recall: 0.6849124244596948
F1-score: 0.7665447709009205
ROC-AUC: 0.9302592403539807

Train set metrics:
Accuracy: 0.8965862485941636
Precision: 0.8685314320946914
Recall: 0.6849241895056909
F1-score: 0.7658772853450752
ROC-AUC: 0.9289758267478305

Validation set metrics:
Accuracy: 0.8962789706538945
Precision: 0.8690252065683506
Recall: 0.6828782181676221
F1-score: 0.7647877762820292
ROC-AUC: 0.9289432357226507


# LightGBM

In [62]:
train_data = lgb.Dataset(train_new.drop(columns=[target1, target2]), label=train_new[target1])
val_data = lgb.Dataset(val_new.drop(columns=[target1, target2]), label=val_new[target1], reference=train_data)

params = {
    'objective': 'binary',           # Задача бинарной классификации
    'boosting_type': 'gbdt',         # Тип бустинга
    'learning_rate': 0.05,           # Скорость обучения
    'num_leaves': 31,                # Количество листьев в дереве решений
    'max_depth': -1,                 # Максимальная глубина дерева (-1 — не ограничено)
    'verbose': -1,                   
    'random_state': 42
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000
)

In [63]:
y_pred_prob = model.predict(test_new.drop(columns=[target1, target2]), num_iteration=model.best_iteration)
y_pred = (y_pred_prob > 0.5).astype(int)  # Преобразование вероятностей в метки классов

accuracy = accuracy_score(test_new[target1], y_pred)
print(f'Accuracy на тесте: {accuracy:.4f}')
print(classification_report(test_new[target1], y_pred))

y_pred_prob = model.predict(test_new.drop(columns=[target1, target2]), num_iteration=model.best_iteration)

roc_auc = roc_auc_score(test_new[target1], y_pred_prob)
print(f'ROC-AUC Score на тесте: {roc_auc:.4f}')

Accuracy на тесте: 0.9205
              precision    recall  f1-score   support

           0       0.92      0.97      0.95    148281
           1       0.91      0.76      0.82     48815

    accuracy                           0.92    197096
   macro avg       0.92      0.87      0.89    197096
weighted avg       0.92      0.92      0.92    197096

ROC-AUC Score на тесте: 0.9647
